# H121 footprint and peatland-QC ObsFcstAna maps

This notebook compares the first six months (2015-04 through 2015-09) of the original H121 DA experiment, the 12.5 km footprint experiment, and the 12.5 km footprint plus peatland-QC experiment. Maps combine the assimilated H SAF Metop-A/B species using their observation counts. Differences are shown as FOV12.5 - H121, peat QC - FOV12.5, and peat QC - H121.

The O-mean and O-F-standard-deviation differences use the common finite tile support for each pair, but the underlying observations are not matched cycle by cycle. They therefore include effects from changed sampling as well as changes in the DA trajectory.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap
from netCDF4 import Dataset

import cartopy.crs as ccrs
import cartopy.feature as cfeature

ROOT = Path.cwd()
while ROOT.name != 'geosldas-analysis' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT.name != 'geosldas-analysis':
    ROOT = Path('/Users/amfox/Desktop/geosldas-analysis')

PROJECT_ROOT = ROOT / 'projects' / 'obs_scaling_params'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from obs_scaling.tile_io import read_tilecoord

DATA_DIR = ROOT / 'data' / 'omf_compare_sums' / 'stats_first6mo'
TILECOORD = PROJECT_ROOT / 'test_data' / 'inputs' / 'OLv7_M36_MULTI_type_13_H121.ldas_tilecoord.bin'

MAP_LAT_MIN = -60.0
MAP_LAT_MAX = 85.0
NMIN = 20
SPECIES_INDICES = (7, 8)  # H SAF Metop-A and Metop-B
RELATIVE_GROUPS = {
    'SMAP': (0, 1, 2, 3),
    'H SAF': SPECIES_INDICES,
}
LAND_FACE = '0.88'
ZERO_NEUTRAL_FRACTION = 0.02

RUNS = {
    'DA_H121': {
        'label': 'H121',
        'file': 'temporal_stats_DA_H121_20150401_20150930.nc4',
    },
    'DA_baseline_FOV12p5': {
        'label': 'FOV12.5',
        'file': 'temporal_stats_DA_baseline_FOV12p5_20150401_20150930.nc4',
    },
    'DA_peatlandqc': {
        'label': 'FOV12.5 + peat QC',
        'file': 'temporal_stats_DA_peatlandqc_20150401_20150930.nc4',
    },
}

OL_REFERENCE = {
    'label': 'OL matched to H121 observations',
    'file': 'temporal_stats_OL_vs_H121obs_20150401_20150930.nc4',
}

COMPARISONS = [
    ('DA_baseline_FOV12p5', 'DA_H121', 'FOV12.5 - H121'),
    ('DA_peatlandqc', 'DA_baseline_FOV12p5', 'Peat QC - FOV12.5'),
    ('DA_peatlandqc', 'DA_H121', 'Peat QC - H121'),
]

mpl.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.titlesize': 10,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

DATA_DIR

In [ ]:
def _as_array(variable):
    raw = variable[:]
    array = np.ma.filled(raw, np.nan) if np.ma.isMaskedArray(raw) else np.asarray(raw)
    array = np.asarray(array, dtype=float)
    fill = getattr(variable, '_FillValue', None)
    if fill is not None:
        array = np.where(array == fill, np.nan, array)
    array = np.where(np.abs(array) < 1.0e14, array, np.nan)
    return array


def load_temporal(path: Path) -> dict[str, np.ndarray]:
    with Dataset(path) as dataset:
        return {name: _as_array(variable) for name, variable in dataset.variables.items()}


def pool_hsaf_species(data: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    indices = list(SPECIES_INDICES)
    n_raw = data['N_data'][:, indices]
    n = np.where(np.isfinite(n_raw) & (n_raw > 0), n_raw, 0.0)
    count = n.sum(axis=1)

    o_values = data['O_mean'][:, indices]
    valid_o = np.isfinite(o_values) & (n > 0)
    n_o = np.where(valid_o, n, 0.0)
    total_o = n_o.sum(axis=1)
    o_mean = np.divide(
        np.where(valid_o, n * o_values, 0.0).sum(axis=1),
        total_o,
        out=np.full(count.shape, np.nan),
        where=total_o >= NMIN,
    )

    omf_mean_values = data['OmF_mean'][:, indices]
    omf_std_values = data['OmF_stdv'][:, indices]
    valid_omf = np.isfinite(omf_mean_values) & np.isfinite(omf_std_values) & (n > 0)
    n_omf = np.where(valid_omf, n, 0.0)
    total_omf = n_omf.sum(axis=1)
    pooled_omf_mean = np.divide(
        np.where(valid_omf, n * omf_mean_values, 0.0).sum(axis=1),
        total_omf,
        out=np.full(count.shape, np.nan),
        where=total_omf >= NMIN,
    )
    pooled_second = np.divide(
        np.where(valid_omf, n * (omf_std_values**2 + omf_mean_values**2), 0.0).sum(axis=1),
        total_omf,
        out=np.full(count.shape, np.nan),
        where=total_omf >= NMIN,
    )
    omf_std = np.sqrt(np.maximum(pooled_second - pooled_omf_mean**2, 0.0))

    return {
        'count': count,
        'count_map': np.where(count > 0, count, np.nan),
        'o_mean': o_mean,
        'omf_std': omf_std,
    }


tilecoord = read_tilecoord(TILECOORD)
lon = tilecoord.com_lon.astype(float)
lat = tilecoord.com_lat.astype(float)
tile_area = tilecoord.area.astype(float)

data = {}
maps = {}
for run_name, config in RUNS.items():
    data[run_name] = load_temporal(DATA_DIR / config['file'])
    assert data[run_name]['N_data'].shape == (tilecoord.n_tile, 10)
    maps[run_name] = pool_hsaf_species(data[run_name])

ol_reference_data = load_temporal(DATA_DIR / OL_REFERENCE['file'])
assert ol_reference_data['N_data'].shape == (tilecoord.n_tile, 10)

tilecoord.n_tile

In [ ]:
def area_mean(values):
    values = np.asarray(values, dtype=float)
    valid = (
        np.isfinite(values)
        & np.isfinite(tile_area)
        & (tile_area > 0)
        & np.isfinite(lat)
        & (lat >= MAP_LAT_MIN)
    )
    if not valid.any():
        return np.nan
    return float(np.sum(values[valid] * tile_area[valid]) / np.sum(tile_area[valid]))


summary = pd.DataFrame([
    {
        'experiment': RUNS[run_name]['label'],
        'total observations': int(np.nansum(run_maps['count'])),
        'tiles with observations': int(np.isfinite(run_maps['count_map']).sum()),
        'mean O': area_mean(run_maps['o_mean']),
        'mean O-F std': area_mean(run_maps['omf_std']),
    }
    for run_name, run_maps in maps.items()
])
summary

In [ ]:
METRICS = {
    'count': {
        'map_key': 'count_map',
        'title': 'H SAF observation count',
        'label': 'Observation count',
        'cmap': 'viridis',
        'absolute_percentiles': (0.0, 99.0),
        'difference_percentile': 99.0,
        'format': '{:.0f}',
    },
    'o_mean': {
        'map_key': 'o_mean',
        'title': 'H SAF O mean',
        'label': 'O mean (m3 m-3)',
        'cmap': 'YlGnBu',
        'absolute_percentiles': (1.0, 99.0),
        'difference_percentile': 99.0,
        'format': '{:.3f}',
    },
    'omf_std': {
        'map_key': 'omf_std',
        'title': 'H SAF O-F standard deviation',
        'label': 'O-F standard deviation (m3 m-3)',
        'cmap': 'magma',
        'absolute_percentiles': (1.0, 99.0),
        'difference_percentile': 99.0,
        'format': '{:.3f}',
    },
}


def panel_label(index: int) -> str:
    return f'({chr(ord("a") + index)})'


def segmented_linear(cmap_name: str, vmin: float, vmax: float, n_bins: int = 12):
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = 0.0, 1.0
    bounds = np.linspace(vmin, vmax, n_bins + 1)
    cmap = plt.get_cmap(cmap_name, n_bins)
    return cmap, BoundaryNorm(bounds, cmap.N, clip=True), bounds


def segmented_diverging(vlim: float, n_side: int = 6):
    vlim = max(float(vlim), np.finfo(float).eps)
    neutral = min(vlim * ZERO_NEUTRAL_FRACTION, vlim * 0.5)
    negative = np.linspace(-vlim, -neutral, n_side + 1)
    positive = np.linspace(neutral, vlim, n_side + 1)
    bounds = np.r_[negative, positive]
    base = plt.get_cmap('RdBu_r', 2 * n_side)
    base_colors = base(np.linspace(0, 1, 2 * n_side))
    colors = np.vstack([
        base_colors[:n_side],
        np.array([[1.0, 1.0, 1.0, 1.0]]),
        base_colors[n_side:],
    ])
    cmap = ListedColormap(colors)
    return cmap, BoundaryNorm(bounds, cmap.N, clip=True), bounds


def decorate_axis(ax):
    ax.set_extent((-180, 180, MAP_LAT_MIN, MAP_LAT_MAX), crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor=LAND_FACE, edgecolor='none', zorder=0)
    ax.coastlines(linewidth=0.4, color='0.3', zorder=3)


def map_scatter(ax, values, cmap, norm):
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(values) & np.isfinite(lon) & np.isfinite(lat) & (lat >= MAP_LAT_MIN)
    artist = ax.scatter(
        lon[valid], lat[valid], c=values[valid], cmap=cmap, norm=norm,
        s=1.15, linewidths=0, rasterized=True, transform=ccrs.PlateCarree(), zorder=2,
    )
    decorate_axis(ax)
    return artist


def add_panel_label(ax, index):
    ax.text(
        0.02, 0.98, panel_label(index), transform=ax.transAxes, ha='left', va='top',
        fontweight='bold', bbox=dict(facecolor='white', edgecolor='none', alpha=0.78, pad=1.5),
        zorder=10,
    )


def plot_absolute(metric_name: str):
    config = METRICS[metric_name]
    values_by_run = [maps[name][config['map_key']] for name in RUNS]
    finite = np.concatenate([value[np.isfinite(value)] for value in values_by_run])
    low_pct, high_pct = config['absolute_percentiles']
    vmin = 0.0 if metric_name == 'count' else float(np.nanpercentile(finite, low_pct))
    vmax = float(np.nanpercentile(finite, high_pct))
    cmap, norm, bounds = segmented_linear(config['cmap'], vmin, vmax)

    fig, axes = plt.subplots(
        1, 3, figsize=(15.2, 4.8), constrained_layout=True,
        subplot_kw={'projection': ccrs.Robinson()},
    )
    artist = None
    for index, (ax, run_name, values) in enumerate(zip(axes, RUNS, values_by_run)):
        artist = map_scatter(ax, values, cmap, norm)
        mean_value = area_mean(values)
        ax.set_title(f"{RUNS[run_name]['label']}\nmean={config['format'].format(mean_value)}")
        add_panel_label(ax, index)
    cbar = fig.colorbar(artist, ax=axes, orientation='horizontal', shrink=0.72, pad=0.06, boundaries=bounds)
    cbar.set_label(config['label'])
    fig.suptitle(config['title'], fontsize=12)
    plt.show()


def plot_differences(metric_name: str):
    config = METRICS[metric_name]
    difference_key = 'count' if metric_name == 'count' else config['map_key']
    differences = [maps[new][difference_key] - maps[old][difference_key] for new, old, _ in COMPARISONS]
    finite_abs = np.concatenate([np.abs(value[np.isfinite(value)]) for value in differences])
    vlim = float(np.nanpercentile(finite_abs, config['difference_percentile']))
    cmap, norm, bounds = segmented_diverging(vlim)

    fig, axes = plt.subplots(
        1, 3, figsize=(15.2, 4.8), constrained_layout=True,
        subplot_kw={'projection': ccrs.Robinson()},
    )
    artist = None
    for index, (ax, difference, comparison) in enumerate(zip(axes, differences, COMPARISONS)):
        artist = map_scatter(ax, difference, cmap, norm)
        mean_value = area_mean(difference)
        ax.set_title(f"{comparison[2]}\nmean={config['format'].format(mean_value)}")
        add_panel_label(ax, index)
    cbar = fig.colorbar(artist, ax=axes, orientation='horizontal', shrink=0.72, pad=0.06, boundaries=bounds)
    ticks = np.linspace(-vlim, vlim, 5)
    cbar.set_ticks(ticks)
    cbar.set_ticklabels([config['format'].format(value) for value in ticks])
    cbar.set_label(f"Difference in {config['label'].lower()}")
    fig.suptitle(f"Difference in {config['title']}", fontsize=12)
    plt.show()

## Figure 1: H SAF observation counts

Six-month Metop-A/B observation counts for each DA experiment. Brighter colors indicate more accepted observations; this is observation support, not a measure of better skill.

In [ ]:
plot_absolute('count')

## Figure 2: differences in H SAF observation counts

Pairwise count differences isolate the wider-footprint effect, the peat-QC effect, and their combined effect. Red means the experiment named first accepted more observations; blue means it accepted fewer; white is near zero.

In [ ]:
plot_differences('count')

## Figure 3: H SAF O mean

Six-month mean scaled H SAF observations, pooled across Metop-A/B. Darker colors indicate wetter mean soil moisture; higher or lower is not intrinsically better.

In [ ]:
plot_absolute('o_mean')

## Figure 4: differences in H SAF O mean

Pairwise differences in scaled observation mean. Red means the experiment named first has a wetter O mean; blue means it has a drier O mean; white is near zero. Differences are restricted to common finite tiles but are not cycle-matched.

In [ ]:
plot_differences('o_mean')

## Figure 5: H SAF O-F standard deviation

Six-month O-F spread pooled across Metop-A/B using the component counts, means, and variances. Lower values indicate a tighter fit to the assimilated observations, although sampling differences must be considered.

In [ ]:
plot_absolute('omf_std')

## Figure 6: differences in H SAF O-F standard deviation

Pairwise differences in O-F spread. Blue/negative means the experiment named first has lower O-F standard deviation (tighter fit); red/positive means higher spread; white is near zero. Differences use common finite tiles but not cycle-matched observations.

In [ ]:
plot_differences('omf_std')

## Figure 7: experiment-first full-period O-F stddev relative-difference maps

Full-period `(experiment - OL) / OL * 100` by tile, using `OL_vs_H121obs` throughout so H SAF observations are on the same scaled basis. SMAP is monitor-only and directly comparable; H SAF results for FOV12.5 and peat QC are approximate because those observations are not cycle-matched to this OL reference. Blue/negative means lower O-F stddev than OL; red/positive means higher; white is near zero.

In [ ]:
def weighted_group_std(data, indices):
    idx = list(indices)
    values = data['OmF_stdv'][:, idx].astype(float)
    weights = data['N_data'][:, idx].astype(float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights >= NMIN)
    numerator = np.where(valid, values * weights, 0.0).sum(axis=1)
    denominator = np.where(valid, weights, 0.0).sum(axis=1)
    return np.divide(
        numerator, denominator, out=np.full(numerator.shape, np.nan), where=denominator > 0,
    )


relative_maps = {}
for run_name in RUNS:
    for group_name, indices in RELATIVE_GROUPS.items():
        experiment_std = weighted_group_std(data[run_name], indices)
        ol_std = weighted_group_std(ol_reference_data, indices)
        relative_maps[(run_name, group_name)] = np.divide(
            100.0 * (experiment_std - ol_std),
            ol_std,
            out=np.full(ol_std.shape, np.nan),
            where=np.isfinite(experiment_std) & np.isfinite(ol_std) & (np.abs(ol_std) > 0),
        )

finite_abs = np.concatenate([
    np.abs(values[np.isfinite(values)]) for values in relative_maps.values()
])
vlim = float(np.clip(np.nanpercentile(finite_abs, 97.5), 15.0, 30.0))
cmap, norm, bounds = segmented_diverging(vlim)

fig, axes = plt.subplots(
    len(RUNS), len(RELATIVE_GROUPS), figsize=(12.8, 9.0), constrained_layout=True,
    subplot_kw={'projection': ccrs.Robinson()},
)
artist = None
panel_index = 0
for row, run_name in enumerate(RUNS):
    for column, group_name in enumerate(RELATIVE_GROUPS):
        ax = axes[row, column]
        values = relative_maps[(run_name, group_name)]
        artist = map_scatter(ax, values, cmap, norm)
        ax.set_title(
            f"{RUNS[run_name]['label']} - OL: {group_name}\n"
            f"mean={area_mean(values):.2f}%"
        )
        add_panel_label(ax, panel_index)
        panel_index += 1

cbar = fig.colorbar(
    artist, ax=axes, orientation='horizontal', shrink=0.66, pad=0.04, boundaries=bounds,
)
ticks = np.linspace(-vlim, vlim, 5)
cbar.set_ticks(ticks)
cbar.set_ticklabels([f'{value:.0f}' for value in ticks])
cbar.set_label('O-F standard deviation relative difference (%)')
fig.suptitle('Full-period O-F standard deviation relative to scaled-observation OL', fontsize=12)
plt.show()

# Canada and Alaska zoom

Regional repeats for FOV12.5 and FOV12.5 + peat QC only.

In [ ]:
ZOOM_RUNS = ('DA_baseline_FOV12p5', 'DA_peatlandqc')
ZOOM_COMPARISON = ('DA_peatlandqc', 'DA_baseline_FOV12p5', 'Peat QC - FOV12.5')
CANADA_ALASKA_EXTENT = (-170.0, -50.0, 40.0, 82.0)


def canada_alaska_mask(values=None):
    west, east, south, north = CANADA_ALASKA_EXTENT
    mask = (lon >= west) & (lon <= east) & (lat >= south) & (lat <= north)
    if values is not None:
        mask &= np.isfinite(np.asarray(values, dtype=float))
    return mask


def regional_area_mean(values):
    values = np.asarray(values, dtype=float)
    valid = canada_alaska_mask(values) & np.isfinite(tile_area) & (tile_area > 0)
    if not valid.any():
        return np.nan
    return float(np.sum(values[valid] * tile_area[valid]) / np.sum(tile_area[valid]))


def regional_finite(values_by_panel):
    selected = []
    for values in values_by_panel:
        values = np.asarray(values, dtype=float)
        selected.append(values[canada_alaska_mask(values)])
    return np.concatenate(selected)


def regional_map_scatter(ax, values, cmap, norm):
    values = np.asarray(values, dtype=float)
    valid = canada_alaska_mask(values)
    artist = ax.scatter(
        lon[valid], lat[valid], c=values[valid], cmap=cmap, norm=norm,
        s=3.0, linewidths=0, rasterized=True, transform=ccrs.PlateCarree(), zorder=2,
    )
    ax.set_extent(CANADA_ALASKA_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor=LAND_FACE, edgecolor='none', zorder=0)
    ax.add_feature(cfeature.BORDERS, linewidth=0.45, edgecolor='0.3', zorder=3)
    ax.coastlines(linewidth=0.5, color='0.3', zorder=3)
    return artist

## Figure 8: Canada and Alaska H SAF observation counts

Full-period H SAF observation counts for FOV12.5 and peat QC. Lighter-to-yellow colors indicate more retained observations.

In [ ]:
count_values = [maps[run_name]['count_map'] for run_name in ZOOM_RUNS]
finite = regional_finite(count_values)
vmin = 0.0
vmax = float(np.nanpercentile(finite, METRICS['count']['absolute_percentiles'][1]))
cmap, norm, bounds = segmented_linear(METRICS['count']['cmap'], vmin, vmax)

fig, axes = plt.subplots(
    1, 2, figsize=(12.8, 5.0), constrained_layout=True,
    subplot_kw={'projection': ccrs.Robinson()},
)
artist = None
for index, (ax, run_name, values) in enumerate(zip(axes, ZOOM_RUNS, count_values)):
    artist = regional_map_scatter(ax, values, cmap, norm)
    ax.set_title(f"{RUNS[run_name]['label']}\nmean={regional_area_mean(values):.0f}")
    add_panel_label(ax, index)
cbar = fig.colorbar(artist, ax=axes, orientation='horizontal', shrink=0.72, pad=0.06, boundaries=bounds)
cbar.set_label('Observation count')
fig.suptitle('Canada and Alaska H SAF observation count', fontsize=12)
plt.show()

## Figure 9: Canada and Alaska difference in H SAF observation counts

Peat QC minus FOV12.5. Blue/negative means peat QC retains fewer observations; red/positive means more; white is near zero.

In [ ]:
count_difference = maps[ZOOM_COMPARISON[0]]['count'] - maps[ZOOM_COMPARISON[1]]['count']
finite_abs = np.abs(regional_finite([count_difference]))
vlim = float(np.nanpercentile(finite_abs, METRICS['count']['difference_percentile']))
cmap, norm, bounds = segmented_diverging(vlim)

fig, ax = plt.subplots(
    1, 1, figsize=(8.8, 5.2), constrained_layout=True,
    subplot_kw={'projection': ccrs.Robinson()},
)
artist = regional_map_scatter(ax, count_difference, cmap, norm)
ax.set_title(f"{ZOOM_COMPARISON[2]}\nmean={regional_area_mean(count_difference):.0f}")
add_panel_label(ax, 0)
cbar = fig.colorbar(artist, ax=ax, orientation='horizontal', shrink=0.74, pad=0.06, boundaries=bounds)
ticks = np.linspace(-vlim, vlim, 5)
cbar.set_ticks(ticks)
cbar.set_ticklabels([f'{value:.0f}' for value in ticks])
cbar.set_label('Difference in observation count')
fig.suptitle('Canada and Alaska difference in H SAF observation count', fontsize=12)
plt.show()

## Figure 10: Canada and Alaska H SAF mean O

Full-period mean scaled H SAF observations. Darker blue indicates a higher (wetter) mean observation.

In [ ]:
o_mean_values = [maps[run_name]['o_mean'] for run_name in ZOOM_RUNS]
finite = regional_finite(o_mean_values)
low_pct, high_pct = METRICS['o_mean']['absolute_percentiles']
vmin = float(np.nanpercentile(finite, low_pct))
vmax = float(np.nanpercentile(finite, high_pct))
cmap, norm, bounds = segmented_linear(METRICS['o_mean']['cmap'], vmin, vmax)

fig, axes = plt.subplots(
    1, 2, figsize=(12.8, 5.0), constrained_layout=True,
    subplot_kw={'projection': ccrs.Robinson()},
)
artist = None
for index, (ax, run_name, values) in enumerate(zip(axes, ZOOM_RUNS, o_mean_values)):
    artist = regional_map_scatter(ax, values, cmap, norm)
    ax.set_title(f"{RUNS[run_name]['label']}\nmean={regional_area_mean(values):.3f}")
    add_panel_label(ax, index)
cbar = fig.colorbar(artist, ax=axes, orientation='horizontal', shrink=0.72, pad=0.06, boundaries=bounds)
cbar.set_label('O mean (m3 m-3)')
fig.suptitle('Canada and Alaska H SAF mean O', fontsize=12)
plt.show()

## Figure 11: Canada and Alaska difference in H SAF mean O

Peat QC minus FOV12.5. Blue/negative means a lower mean observation with peat QC; red/positive means higher; white is near zero.

In [ ]:
o_mean_difference = maps[ZOOM_COMPARISON[0]]['o_mean'] - maps[ZOOM_COMPARISON[1]]['o_mean']
finite_abs = np.abs(regional_finite([o_mean_difference]))
vlim = float(np.nanpercentile(finite_abs, METRICS['o_mean']['difference_percentile']))
cmap, norm, bounds = segmented_diverging(vlim)

fig, ax = plt.subplots(
    1, 1, figsize=(8.8, 5.2), constrained_layout=True,
    subplot_kw={'projection': ccrs.Robinson()},
)
artist = regional_map_scatter(ax, o_mean_difference, cmap, norm)
ax.set_title(f"{ZOOM_COMPARISON[2]}\nmean={regional_area_mean(o_mean_difference):.2e}")
add_panel_label(ax, 0)
cbar = fig.colorbar(artist, ax=ax, orientation='horizontal', shrink=0.74, pad=0.06, boundaries=bounds)
ticks = np.linspace(-vlim, vlim, 5)
cbar.set_ticks(ticks)
cbar.set_ticklabels([f'{value:.1e}' for value in ticks])
cbar.set_label('Difference in O mean (m3 m-3)')
fig.suptitle('Canada and Alaska difference in H SAF mean O', fontsize=12)
plt.show()

## Figure 12: Canada and Alaska O-F stddev relative-difference maps

Regional repeat of Figure 7 for FOV12.5 and peat QC, using `OL_vs_H121obs`. Blue/negative means lower O-F stddev than OL; red/positive means higher; white is near zero. SMAP is directly comparable; H SAF is not cycle-matched to this OL reference.

In [ ]:
regional_relative_values = [
    relative_maps[(run_name, group_name)]
    for run_name in ZOOM_RUNS
    for group_name in RELATIVE_GROUPS
]
finite_abs = np.abs(regional_finite(regional_relative_values))
vlim = float(np.clip(np.nanpercentile(finite_abs, 97.5), 15.0, 30.0))
cmap, norm, bounds = segmented_diverging(vlim)

fig, axes = plt.subplots(
    len(ZOOM_RUNS), len(RELATIVE_GROUPS), figsize=(12.8, 7.2), constrained_layout=True,
    subplot_kw={'projection': ccrs.Robinson()},
)
artist = None
panel_index = 0
for row, run_name in enumerate(ZOOM_RUNS):
    for column, group_name in enumerate(RELATIVE_GROUPS):
        ax = axes[row, column]
        values = relative_maps[(run_name, group_name)]
        artist = regional_map_scatter(ax, values, cmap, norm)
        ax.set_title(
            f"{RUNS[run_name]['label']} - OL: {group_name}\n"
            f"mean={regional_area_mean(values):.2f}%"
        )
        add_panel_label(ax, panel_index)
        panel_index += 1
cbar = fig.colorbar(artist, ax=axes, orientation='horizontal', shrink=0.66, pad=0.05, boundaries=bounds)
ticks = np.linspace(-vlim, vlim, 5)
cbar.set_ticks(ticks)
cbar.set_ticklabels([f'{value:.0f}' for value in ticks])
cbar.set_label('O-F standard deviation relative difference (%)')
fig.suptitle('Canada and Alaska O-F standard deviation relative to scaled-observation OL', fontsize=12)
plt.show()